# 01 — Benchmark Frugalité CPU (1 thread, ONNX INT8)

Notebook 100 % autonome (aucun `HF_TOKEN`, aucun Git requis) :
1. Télécharge 1 archive audio réelle depuis le bucket public `https://storage.googleapis.com/fongbe/`.
2. Vérifie la parité de l'extracteur pur NumPy (`features_numpy.py`) avec `SeamlessM4TFeatureExtractor`.
3. Exporte `facebook/w2v-bert-2.0` (MIT) en **ONNX INT8** pour **8, 12, 16 et 24 couches**.
4. Mesure **taille / RTF / pic RAM** sur 1 seul cœur CPU avec le script officiel `measure_frugality.py`.

**Réglages Kaggle** : `Accelerator: None (CPU)` + `Internet: ON`.


In [ ]:
# 1. Installer les bibliothèques nécessaires
!pip install -q "transformers==5.14.1" soundfile onnx onnxruntime

In [ ]:
# 2. Lister le bucket GCS officiel et télécharger 1 archive audio (~2 Go)
import csv, tarfile, urllib.parse, urllib.request, xml.etree.ElementTree as ET
from pathlib import Path

BUCKET = "https://storage.googleapis.com/fongbe"
DATA_DIR = Path("/tmp/fongbe")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def list_bucket():
    items, marker = [], ""
    while True:
        url = f"{BUCKET}/" + (f"?marker={urllib.parse.quote(marker)}" if marker else "")
        with urllib.request.urlopen(url, timeout=60) as r:
            root = ET.fromstring(r.read())
        trunc, last = False, ""
        for e in root:
            t = e.tag.split("}")[-1]
            if t == "IsTruncated" and (e.text or "").lower() == "true":
                trunc = True
            elif t == "Contents":
                k, s = "", 0
                for c in e:
                    ct = c.tag.split("}")[-1]
                    if ct == "Key": k = c.text or ""
                    elif ct == "Size": s = int(c.text or 0)
                if k:
                    items.append((k, s)); last = k
        if not trunc or not last:
            break
        marker = last
    return items

items = list_bucket()
print(f"Bucket GCS: {len(items)} fichiers ({sum(s for _, s in items)/1e9:.2f} GB total)")
for k, s in items:
    print(f"  {s/1e6:8.1f} MB  {k}")

# Télécharge et extrait la 1ère archive train_reliable (.tar)
rel_tars = [k for k, _ in items if "reliable" in k and k.endswith(".tar")]
assert rel_tars, "Aucune archive reliable trouvée dans le bucket!"
tar_key = rel_tars[0]
marker = DATA_DIR / f".done_{Path(tar_key).name}"
if not marker.exists():
    tar_path = DATA_DIR / Path(tar_key).name
    print(f"\nTéléchargement de {tar_key} ...")
    urllib.request.urlretrieve(f"{BUCKET}/{urllib.parse.quote(tar_key)}", str(tar_path))
    print("Extraction ...")
    with tarfile.open(tar_path) as tar:
        tar.extractall(DATA_DIR)
    tar_path.unlink()
    marker.touch()
    print("Extraction terminée !")
else:
    print(f"Déjà extrait : {tar_key}")

wavs = sorted(DATA_DIR.rglob("*.wav"))
print(f"WAV disponibles : {len(wavs):,}")

# Crée un CSV de 50 audios réels pour mesurer RTF + pic RAM
WORK = Path("/kaggle/working/frugality_bench") if Path("/kaggle/working").exists() else Path("frugality_bench")
WORK.mkdir(parents=True, exist_ok=True)
bench_csv = WORK / "bench_50.csv"
with open(bench_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "audio_path"])
    for p in wavs[:50]:
        w.writerow([p.stem, p.relative_to(DATA_DIR).as_posix()])
print("CSV de benchmark écrit :", bench_csv)

In [ ]:
# 3. Écrire les scripts autonomes (features_numpy.py, infer_onnx.py, export_onnx_int8.py, measure_frugality.py)
import json
from pathlib import Path

WORK = Path("/kaggle/working/frugality_bench") if Path("/kaggle/working").exists() else Path("frugality_bench")
WORK.mkdir(parents=True, exist_ok=True)

# Vocabulaire officiel NFD (29 lettres + 5 tons + | + [UNK] + [PAD] = 37 tokens)
letters = sorted(set("abcdefghijklmnopqrstuvwxyz") | {"ɔ", "ɖ", "ɛ"})
tones = sorted({"\u0301", "\u0300", "\u030c", "\u0302", "\u0304"})
vocab = {c: i for i, c in enumerate(letters + tones)}
vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)
(WORK / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=2), encoding="utf-8")

FEATURES_PY = '"""Pure-NumPy Kaldi log-Mel + stride-2 feature extractor for facebook/w2v-bert-2.0.\n\nZero PyTorch / Transformers dependency at inference time (saves ~400-500 MB RAM\nand several seconds of startup time under `measure_frugality.py`).\n\nMatches Hugging Face `SeamlessM4TFeatureExtractor` (80 Kaldi mel bins, 16 kHz,\n25 ms frame / 10 ms shift, Povey window, preemphasis 0.97, utterance CMVN,\nstride-2 stacking -> 160-dim features). Parity is checked automatically by\n`src/export_onnx_int8.py`.\n"""\nfrom __future__ import annotations\n\nimport math\nimport numpy as np\n\n\ndef _mel_scale(freq: np.ndarray) -> np.ndarray:\n    return 1127.0 * np.log(1.0 + freq / 700.0)\n\n\ndef _kaldi_mel_filterbank(\n    num_mel_bins: int = 80,\n    n_fft: int = 512,\n    sample_rate: int = 16000,\n    low_freq: float = 20.0,\n    high_freq: float = 8000.0,\n) -> np.ndarray:\n    """Kaldi triangular mel filterbank matrix of shape (n_fft // 2 + 1, num_mel_bins)."""\n    num_fft_bins = n_fft // 2\n    fft_bin_width = sample_rate / n_fft\n    mel_low = float(_mel_scale(np.array(low_freq)))\n    mel_high = float(_mel_scale(np.array(high_freq)))\n    mel_delta = (mel_high - mel_low) / (num_mel_bins + 1)\n\n    bins = np.arange(num_mel_bins, dtype=np.float64)\n    left_mel = mel_low + bins * mel_delta\n    center_mel = mel_low + (bins + 1.0) * mel_delta\n    right_mel = mel_low + (bins + 2.0) * mel_delta\n\n    freqs = np.arange(num_fft_bins, dtype=np.float64) * fft_bin_width\n    mel = _mel_scale(freqs)[:, None]  # (num_fft_bins, 1)\n\n    up_slope = (mel - left_mel[None, :]) / (center_mel[None, :] - left_mel[None, :])\n    down_slope = (right_mel[None, :] - mel) / (right_mel[None, :] - center_mel[None, :])\n    fb = np.maximum(0.0, np.minimum(up_slope, down_slope))\n\n    # Append Nyquist bin (zeroed, matching Kaldi / HF audio_utils)\n    fb = np.vstack([fb, np.zeros((1, num_mel_bins), dtype=np.float64)])\n    return fb.astype(np.float32)\n\n\n_MEL_FB_80 = _kaldi_mel_filterbank(num_mel_bins=80, n_fft=512, sample_rate=16000, low_freq=20.0, high_freq=8000.0)\n_POVEY_400 = np.power(0.5 - 0.5 * np.cos(2.0 * math.pi * np.arange(400, dtype=np.float64) / 399.0), 0.85).astype(\n    np.float32\n)\n\n\ndef extract_w2vbert_features(\n    waveform: np.ndarray,\n    sample_rate: int = 16000,\n    stride: int = 2,\n    padding_value: float = 1.0,\n) -> np.ndarray:\n    """Extract (T_out, 160) float32 features matching SeamlessM4TFeatureExtractor.\n\n    Args:\n        waveform: 1D float32 audio in [-1, 1] at 16 kHz.\n        sample_rate: Must be 16000.\n        stride: Frame stacking stride (2 for w2v-bert-2.0 -> 80 * 2 = 160 dims).\n        padding_value: Pad value if odd frame count (unused when dropping/zero-padding).\n    """\n    if sample_rate != 16000:\n        raise ValueError(f"Expected 16000 Hz audio, got {sample_rate}")\n    wav = np.asarray(waveform, dtype=np.float32)\n    if wav.ndim > 1:\n        wav = wav.mean(axis=-1)\n\n    # SeamlessM4TFeatureExtractor multiplies waveform by 2**15 (Kaldi int16 scale)\n    wav = wav * 32768.0\n\n    frame_length = 400  # 25 ms\n    frame_shift = 160   # 10 ms\n    n_fft = 512\n\n    if len(wav) < frame_length:\n        wav = np.pad(wav, (0, frame_length - len(wav)), mode="constant")\n\n    num_frames = 1 + (len(wav) - frame_length) // frame_shift\n    # Strided framing view (num_frames, 400)\n    itemsize = wav.strides[0]\n    frames = np.lib.stride_tricks.as_strided(\n        wav,\n        shape=(num_frames, frame_length),\n        strides=(frame_shift * itemsize, itemsize),\n    ).copy()\n\n    # Remove DC offset per frame\n    frames -= frames.mean(axis=1, keepdims=True)\n\n    # Pre-emphasis (0.97)\n    preemph = 0.97\n    frames[:, 1:] = frames[:, 1:] - preemph * frames[:, :-1]\n    frames[:, 0] = frames[:, 0] * (1.0 - preemph)\n\n    # Povey window + 512-pt rFFT power spectrum\n    frames *= _POVEY_400[None, :]\n    spec = np.fft.rfft(frames, n=n_fft, axis=1)\n    power = (spec.real ** 2 + spec.imag ** 2).astype(np.float32)\n\n    # Mel filterbank + log\n    mel_energies = np.matmul(power, _MEL_FB_80)\n    log_mel = np.log(np.maximum(mel_energies, np.float32(1.1920929e-7)))\n\n    # Per-utterance zero-mean unit-variance normalisation (ddof=1 matches HF SeamlessM4T)\n    mean = log_mel.mean(axis=0, keepdims=True)\n    var = log_mel.var(axis=0, ddof=1 if len(log_mel) > 1 else 0, keepdims=True)\n    log_mel = (log_mel - mean) / np.sqrt(var + np.float32(1e-7))\n\n    # Stride-2 stacking (SeamlessM4T trims trailing odd frame: `num_frames // stride * stride`)\n    n_stacked = (log_mel.shape[0] // stride) * stride\n    if n_stacked == 0:\n        pad_n = stride - log_mel.shape[0]\n        log_mel = np.pad(log_mel, ((0, pad_n), (0, 0)), constant_values=padding_value)\n        n_stacked = stride\n    stacked = log_mel[:n_stacked].reshape(n_stacked // stride, log_mel.shape[1] * stride)\n    return np.ascontiguousarray(stacked, dtype=np.float32)\n'
(WORK / "features_numpy.py").write_text(FEATURES_PY, encoding="utf-8")

MEASURE_PY = '#!/usr/bin/env python\n"""Frugality measurement of the "Frugal Tonal ASR for Fongbe" competition: the script the organisers run on the\ntop 20 of the private leaderboard, provided so that every team can check its compliance before submitting.\n\n    python measure_frugality.py \\\n        --weights my_system/ --weights lm.bin \\\n        --command "python infer.py --csv {csv} --root {root} --output {output}" \\\n        --csv Test.csv [--limit 300] [--report report.json]\n\nYour command reads a CSV (`id, audio_path`, paths relative to `{root}`) and writes `{output}` (`id, transcription`,\nor the submission format `ID, Target`).\nWhat the script measures, on the complete system, from audio to tonal transcription:\n\n  size        sum of the files given with --weights: ALL the weights (pre-trained models and language model included,\n              after any quantisation), threshold 1,000 MB (MB = 10^6 bytes)\n  RTF         elapsed time of the command / total audio duration, on ONE single CPU core, threshold 1\n  memory      peak resident memory of the command and all its sub-processes, threshold 2 GB (GB = 10^9 bytes)\n  offline     the command runs with the network cut (Linux, root rights: `unshare -n`) and without any GPU\n\nIf the CSV has a `transcription` column (for example the `recommended_validation` voices of Train_reliable.csv),\nthe official score is recomputed with `score.py`. The organisers recompute it on the full test set.\n\nConditions of the official check: recent x86-64 Linux machine, a single thread (CPU affinity + *_NUM_THREADS variables),\nno GPU, network cut; model loading counts in the measured time. The thresholds are those of the competition rules;\nexceeding one excludes a team from the prizes, it does not change the leaderboard.\n"""\nimport argparse\nimport csv\nimport json\nimport os\nimport resource\nimport shlex\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport wave\nfrom pathlib import Path\n\nMAX_SIZE = 1000 * 10 ** 6       # 1,000 MB\nMAX_RTF = 1.0\nMAX_MEMORY = 2 * 10 ** 9        # 2 GB\nTHREADS = {k: "1" for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",\n                            "VECLIB_MAXIMUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS", "ONNXRUNTIME_NUM_THREADS")}\nOFFLINE = {"CUDA_VISIBLE_DEVICES": "", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"}\nNETWORK_NOT_CUT = "network NOT cut"\n\n\ndef weights_size(paths):\n    seen, total, detail = set(), 0, {}\n    for c in paths:\n        p = Path(c)\n        files = [p] if p.is_file() else [f for f in p.rglob("*") if f.is_file()] if p.is_dir() else []\n        if not files:\n            sys.exit(f"--weights not found or empty: {c}")\n        t = 0\n        for f in files:\n            key = f.resolve()\n            if key not in seen:                    # the same file (link) is only counted once\n                seen.add(key); t += f.stat().st_size\n        detail[str(c)] = t; total += t\n    return total, detail\n\n\ndef wav_duration(path):\n    with wave.open(str(path)) as w:\n        return w.getnframes() / w.getframerate()\n\n\ndef tree_memory(pid):\n    """Resident memory (bytes) of the process and all its descendants, read from /proc (Linux)."""\n    total, todo = 0, [pid]\n    while todo:\n        p = todo.pop()\n        try:\n            for line in open(f"/proc/{p}/status"):\n                if line.startswith("VmRSS:"):\n                    total += int(line.split()[1]) * 1024\n                    break\n            for task in os.listdir(f"/proc/{p}/task"):\n                with open(f"/proc/{p}/task/{task}/children") as f:\n                    todo.extend(int(x) for x in f.read().split())\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            continue\n    return total\n\n\ndef run(command, env, one_core, no_network):\n    prefix, notes = [], []\n    if no_network:\n        if shutil.which("unshare") and os.geteuid() == 0:\n            prefix = ["unshare", "-n"]\n        else:\n            notes.append(f"{NETWORK_NOT_CUT} (Linux, `unshare` and root rights are needed): check yourself that no network call is made")\n    core = None\n    if one_core and hasattr(os, "sched_setaffinity"):\n        core = min(os.sched_getaffinity(0))\n    elif one_core:\n        notes.append("single core NOT enforced (Linux needed): only the *_NUM_THREADS variables limit the threads")\n\n    def before():\n        if core is not None:\n            os.sched_setaffinity(0, {core})\n\n    t0 = time.perf_counter()\n    proc = subprocess.Popen(prefix + ["/bin/sh", "-c", command], env=env, preexec_fn=before if os.name == "posix" else None)\n    peak = 0\n    while proc.poll() is None:\n        if os.path.isdir("/proc"):\n            peak = max(peak, tree_memory(proc.pid))\n        time.sleep(0.05)\n    elapsed = time.perf_counter() - t0\n    # complement: the largest descendant process seen by the kernel (ru_maxrss in KiB on Linux)\n    peak = max(peak, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss * 1024)\n    return proc.returncode, elapsed, peak, notes, core\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="Frugality measurement (size, RTF on one core, peak memory, offline).")\n    ap.add_argument("--weights", action="append", required=True, help="weights folder or file; repeat for each one")\n    ap.add_argument("--command", required=True, help="inference command with {csv}, {root} and {output}")\n    ap.add_argument("--csv", required=True, help="CSV with id, audio_path (and transcription to recompute the score)")\n    ap.add_argument("--root", help="folder the audio_path paths are relative to (default: the folder of the CSV)")\n    ap.add_argument("--limit", type=int, help="only measure the first N utterances (quick check)")\n    ap.add_argument("--report", help="JSON file for the report")\n    ap.add_argument("--allow-network", action="store_true", help="do not cut the network (not recommended)")\n    a = ap.parse_args()\n\n    source = Path(a.csv).resolve()\n    root = Path(a.root).resolve() if a.root else source.parent\n    with open(source, encoding="utf-8-sig", newline="") as f:\n        rows = list(csv.DictReader(f))\n    if a.limit:\n        rows = rows[:a.limit]\n    if not rows or not {"id", "audio_path"} <= set(rows[0]):\n        sys.exit("the CSV needs at least the columns id and audio_path")\n    audio_total = sum(wav_duration(root / r["audio_path"]) for r in rows)\n\n    size, detail = weights_size(a.weights)\n    work = Path(tempfile.mkdtemp(prefix="frugality-"))\n    input_csv, output = work / "input.csv", work / "submission.csv"\n    with open(input_csv, "w", encoding="utf-8", newline="") as f:\n        w = csv.writer(f); w.writerow(["id", "audio_path"]); w.writerows((r["id"], r["audio_path"]) for r in rows)\n    command = a.command.format(csv=shlex.quote(str(input_csv)), root=shlex.quote(str(root)), output=shlex.quote(str(output)))\n    env = dict(os.environ, **THREADS, **OFFLINE)\n    print(f"{len(rows)} utterances, {audio_total / 60:.1f} min of audio; weights: {size / 1e6:.1f} MB; running…", flush=True)\n    code, elapsed, peak, notes, core = run(command, env, True, not a.allow_network)\n\n    report = {"utterances": len(rows), "audio_s": round(audio_total, 2), "command": a.command, "return_code": code,\n              "size_bytes": size, "size_detail": detail, "time_s": round(elapsed, 2),\n              "RTF": round(elapsed / audio_total, 4) if audio_total else None, "peak_memory_bytes": peak,\n              "core": core, "threads": THREADS, "notes": notes}\n    if code != 0 or not output.exists():\n        report["error"] = f"the command failed (code {code}) or did not write {output}"\n    else:\n        with open(output, encoding="utf-8-sig", newline="") as f:\n            sub = {(r["ID"] if "ID" in r else r["id"]): (r.get("Target") if "ID" in r else r.get("transcription")) or ""\n                   for r in csv.DictReader(f)}\n        ids = [r["id"] for r in rows]\n        report["submission"] = {"missing": sum(i not in sub for i in ids), "extra": len(set(sub) - set(ids)),\n                                "empty": sum(not sub.get(i, "").strip() for i in ids)}\n        if "transcription" in rows[0]:\n            sys.path.insert(0, str(Path(__file__).resolve().parent))\n            from score import measure\n            s = measure([(r["transcription"], sub.get(r["id"], "")) for r in rows])\n            report["score"] = {k: s[k] for k in ("score", "TSER", "TER_conditional")}\n\n    def verdict(value, limit):\n        return "not measured" if value is None else ("compliant" if value <= limit else "NOT COMPLIANT")\n    report["verdict"] = {"size": verdict(size, MAX_SIZE), "RTF": verdict(report["RTF"], MAX_RTF),\n                         "memory": verdict(peak, MAX_MEMORY),\n                         "offline": "not checked" if any(NETWORK_NOT_CUT in n for n in notes) else "network cut"}\n    print(f"size     {size / 1e6:9.1f} MB   (<= {MAX_SIZE / 1e6:.0f})   {report[\'verdict\'][\'size\']}")\n    print(f"RTF      {report[\'RTF\']:9.3f}      (<= {MAX_RTF:g})       {report[\'verdict\'][\'RTF\']}   [{elapsed:.1f} s for {audio_total:.1f} s of audio]")\n    print(f"memory   {peak / 1e9:9.3f} GB   (<= {MAX_MEMORY / 1e9:.0f})       {report[\'verdict\'][\'memory\']}")\n    print(f"network  {report[\'verdict\'][\'offline\']}")\n    if "score" in report:\n        print(f"score    {report[\'score\'][\'score\']:.4f}  (TSER {report[\'score\'][\'TSER\']:.4f}, conditional TER {report[\'score\'][\'TER_conditional\']:.4f})")\n    for n in notes + ([report["error"]] if "error" in report else []):\n        print("WARNING:", n)\n    if a.report:\n        Path(a.report).write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")\n    shutil.rmtree(work, ignore_errors=True)\n    sys.exit(0 if "error" not in report and all(v in ("compliant", "network cut") for v in report["verdict"].values()) else 1)\n\n\nif __name__ == "__main__":\n    main()\n'
(WORK / "measure_frugality.py").write_text(MEASURE_PY, encoding="utf-8")

INFER_PY = '''import argparse, csv, json, os, sys, wave
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "ONNXRUNTIME_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import numpy as np, onnxruntime as ort
sys.path.insert(0, str(Path(__file__).resolve().parent))
from features_numpy import extract_w2vbert_features

def read_wav_16k(path):
    with wave.open(str(path), "rb") as w:
        raw = w.readframes(w.getnframes())
        ch = w.getnchannels()
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    return audio.reshape(-1, ch).mean(axis=1) if ch > 1 else audio

def greedy(logits, id2tok, pad_id, unk_id):
    ids = np.argmax(logits, axis=-1)
    out, prev = [], -1
    for i in ids:
        ii = int(i)
        if ii != prev and ii != pad_id and ii != unk_id and 0 <= ii < len(id2tok):
            t = id2tok[ii]
            out.append(" " if t == "|" else t)
        prev = ii
    return "".join(out).strip()

ap = argparse.ArgumentParser()
ap.add_argument("--weights", type=Path, required=True)
ap.add_argument("--csv", type=Path, required=True)
ap.add_argument("--root", type=Path, required=True)
ap.add_argument("--output", type=Path, required=True)
a = ap.parse_args()

vocab = json.loads((a.weights / "vocab.json").read_text(encoding="utf-8"))
id2tok = [""] * len(vocab)
for k, v in vocab.items(): id2tok[v] = k
pad_id, unk_id = vocab["[PAD]"], vocab["[UNK]"]

opts = ort.SessionOptions()
opts.intra_op_num_threads = 1; opts.inter_op_num_threads = 1
opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession(str(a.weights / "model_int8.onnx"), sess_options=opts, providers=["CPUExecutionProvider"])

rows = list(csv.DictReader(open(a.csv, encoding="utf-8-sig")))
with open(a.output, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "transcription"])
    for r in rows:
        wav = read_wav_16k(a.root / r["audio_path"])
        feats = extract_w2vbert_features(wav, 16000)[None, :, :]
        mask = np.ones((1, feats.shape[1]), dtype=np.int64)
        logits = sess.run(None, {"input_features": feats, "attention_mask": mask})[0][0]
        w.writerow([r["id"], greedy(logits, id2tok, pad_id, unk_id)])
'''
(WORK / "infer_onnx.py").write_text(INFER_PY, encoding="utf-8")

EXPORT_PY = '''import argparse, shutil, sys, tempfile
from pathlib import Path
import numpy as np, torch
from onnxruntime.quantization import QuantType, quantize_dynamic
from transformers import SeamlessM4TFeatureExtractor, Wav2Vec2BertForCTC, Wav2Vec2CTCTokenizer
sys.path.insert(0, str(Path(__file__).resolve().parent))
from features_numpy import extract_w2vbert_features

class Wrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_features, attention_mask):
        return self.m(input_features=input_features, attention_mask=attention_mask).logits

ap = argparse.ArgumentParser()
ap.add_argument("--num-layers", type=int, required=True)
ap.add_argument("--vocab", type=Path, required=True)
ap.add_argument("--out-dir", type=Path, required=True)
a = ap.parse_args()
a.out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(a.vocab, a.out_dir / "vocab.json")

# Check feature parity once
rng = np.random.default_rng(42)
wav = (0.1 * rng.standard_normal(16000 * 4)).astype(np.float32)
ref = SeamlessM4TFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")(wav, sampling_rate=16000).input_features[0]
ours = extract_w2vbert_features(wav, 16000)
m = min(ref.shape[0], ours.shape[0])
print(f"Feature parity: ref={ref.shape}, numpy={ours.shape}, mean_diff={np.mean(np.abs(ref[:m]-ours[:m])):.6f}, max_diff={np.max(np.abs(ref[:m]-ours[:m])):.6f}")

tok = Wav2Vec2CTCTokenizer(str(a.vocab), unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", bos_token=None, eos_token=None)
model = Wav2Vec2BertForCTC.from_pretrained("facebook/w2v-bert-2.0", add_adapter=True, pad_token_id=tok.pad_token_id, vocab_size=len(tok))
if a.num_layers < len(model.wav2vec2_bert.encoder.layers):
    model.wav2vec2_bert.encoder.layers = model.wav2vec2_bert.encoder.layers[:a.num_layers]
    model.config.num_hidden_layers = a.num_layers
model.eval().cpu()

with tempfile.TemporaryDirectory() as tmp:
    fp32 = Path(tmp) / "fp32.onnx"
    kw = dict(input_names=["input_features", "attention_mask"], output_names=["logits"],
              dynamic_axes={"input_features": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"}, "logits": {0: "b", 1: "ot"}},
              opset_version=17, do_constant_folding=True)
    try:
        torch.onnx.export(Wrap(model), (torch.randn(1, 150, 160), torch.ones(1, 150, dtype=torch.long)), str(fp32), dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(Wrap(model), (torch.randn(1, 150, 160), torch.ones(1, 150, dtype=torch.long)), str(fp32), **kw)
    quantize_dynamic(str(fp32), str(a.out_dir / "model_int8.onnx"), weight_type=QuantType.QInt8)
sz = sum(f.stat().st_size for f in a.out_dir.rglob("*") if f.is_file())
print(f"Exported {a.num_layers}L INT8 -> {sz/1e6:.1f} MB")
'''
(WORK / "export_onnx_int8.py").write_text(EXPORT_PY, encoding="utf-8")
print("Scripts prêts dans :", WORK)

In [ ]:
# 4. Benchmark officiel 1-thread CPU (ONNX INT8) : 8L vs 12L vs 16L vs 24L
import json, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working/frugality_bench") if Path("/kaggle/working").exists() else Path("frugality_bench")
DATA_DIR = Path("/tmp/fongbe")

summary = []
for n_layers in [8, 12, 16, 24]:
    print(f"\n==================== {n_layers} COUCHES (ONNX INT8, 1 CPU) ====================", flush=True)
    w_dir = WORK / f"weights_{n_layers}L"
    rep = WORK / f"report_{n_layers}L.json"
    try:
        subprocess.run([sys.executable, str(WORK / "export_onnx_int8.py"),
                        "--num-layers", str(n_layers), "--vocab", str(WORK / "vocab.json"), "--out-dir", str(w_dir)], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Export échoué pour {n_layers}L (code {e.returncode}) — on continue.")
        continue
    cmd = f"{sys.executable} {WORK / 'infer_onnx.py'} --weights {w_dir} --csv {{csv}} --root {{root}} --output {{output}}"
    subprocess.run([sys.executable, str(WORK / "measure_frugality.py"), "--allow-network",
                    "--weights", str(w_dir), "--command", cmd,
                    "--csv", str(WORK / "bench_50.csv"), "--root", str(DATA_DIR), "--report", str(rep)], check=False)
    if rep.exists():
        d = json.loads(rep.read_text())
        summary.append((n_layers, d["size_bytes"]/1e6, d["RTF"], d["peak_memory_bytes"]/1e9, d["verdict"]))

print("\n================ RÉSUMÉ FRUGALITÉ (limites: <=1000 MB | RTF<=1.0 | <=2.0 GB) ================")
for nl, sz, rtf, mem, v in summary:
    print(f"  w2v-bert-2.0 ({nl:2d}L INT8): size={sz:6.1f} MB ({v['size']}) | RTF={rtf:.3f} ({v['RTF']}) | RAM={mem:.3f} GB ({v['memory']})")